In [24]:
import pandas as pd
# bol corr function from Gaia DR3
import sys
from pathlib import Path
homedir = Path.home()
sys.path.append(str(homedir)+"/"+"gaiadr3_bcg") # navigate to dir containing corr package
import gdr3bcg.bcg as bcg

# **Crossmatch Data with Gaia DR3**

Read out ID, RA, and Dec for cleaned combined catalogue.

In [25]:
full_data = pd.read_csv("../2_duplicatestars/asteroseismic_clean.csv")

idents = pd.DataFrame({
    "gdr3_id": full_data["gaiadr3_id"],
    "ra": full_data["ra"],
    "dec": full_data["dec"]
})

# write to .csv file
idents.to_csv("../0_inputdata_spectroscopic/idents.csv", index=False)

## **Xmatch with Gaia Params + BJ21 Distances**

Paper : https://ui.adsabs.harvard.edu/abs/2021AJ....161..147B/abstract

Catalogue : https://ui.adsabs.harvard.edu/abs/2021yCat.1352....0B/abstract

ADQL Query:

```
SELECT 
lj.gdr3_id AS gdr3_id,
lj.ra AS ra,
lj.dec AS dec,

bj.r_med_geo AS bj_rgeo,
bj.r_lo_geo AS bj_b_rgeo,
bj.r_hi_geo AS bj_B_rgeo,

bj.r_med_photogeo AS bj_rphoto,
bj.r_lo_photogeo AS bj_b_rphoto,
bj.r_hi_photogeo AS bj_B_rphoto,
bj.flag AS bj_flag,

g.phot_g_mean_mag AS gaia_gmag,
g.phot_bp_mean_mag AS gaia_bpmag,
g.phot_rp_mean_mag AS gaia_rpmag,

g.ag_gspphot AS gspphot_ag,
g.ag_gspphot_lower AS gspphot_b_ag,
g.ag_gspphot_upper AS gspphot_B_ag,

g.ruwe AS gaia_ruwe,
g.pmra AS gaia_pmra,
g.pmra_error AS gaia_e_pmra,
g.pmdec AS gaia_pmdec,
g.pmdec_error AS gaia_e_pmdec,

g.parallax AS gaia_plx,
g.parallax_error AS gaia_e_plx,
g.radial_velocity AS gaia_rv,
g.radial_velocity_error AS gaia_e_rv,

g.teff_gspphot AS gspphot_teff,
g.teff_gspphot_lower AS gspphot_b_teff,
g.teff_gspphot_upper AS gspphot_B_teff,
 
g.logg_gspphot AS gspphot_logg,
g.logg_gspphot_lower AS gspphot_b_logg,
g.logg_gspphot_upper AS gspphot_B_logg,

g.distance_gspphot AS gspphot_dist,
g.distance_gspphot_lower AS gspphot_b_dist,
g.distance_gspphot_upper AS gspphot_B_dist

FROM user_lspriggs.idents AS lj
LEFT JOIN external.gaiaedr3_distance AS bj ON lj.gdr3_id = bj.source_id
LEFT JOIN gaiadr3.gaia_source AS g ON lj.gdr3_id = g.source_id
```

Read-in and merge data.

In [ ]:
ext_data = pd.read_csv("../0_inputdata_spectroscopic/gaia_params.csv")

# rename columns
ext_data.columns = ext_data.columns.str.replace("bj_", "BJ-")
ext_data.columns = ext_data.columns.str.replace("gaia_", "GAIA-")
ext_data.columns = ext_data.columns.str.replace("gspphot_", "GSPPHOT-")

# convert lower + upper bounds to single uncertainty
upper_cols = [col for col in ext_data.columns if "-B_" in col] # find all params w upper bound
for upper_col in upper_cols:
    lower_col = upper_col.replace("-B_", "-b_") # find corresponding lower bound
    ext_data[upper_col.replace("-B_", "-e_")] = (ext_data[upper_col] - ext_data[lower_col]) / 2 # compute err value
    ext_data = ext_data.drop(columns=[upper_col, lower_col]) # discard lower + upper bounds

int_data = pd.read_csv("../2_duplicatestars/asteroseismic_clean.csv")

data = pd.merge(int_data, ext_data, how="left", left_on="gaiadr3_id", right_on="gdr3_id")

## Derive Stellar Luminosities

In [ ]:
lum_derive = pd.DataFrame({
    "gaiadr3_id": data["gaiadr3_id"],
    
})

**Account for extinction:**

$$m_{G,\,\mathrm{corr}} = m_{G,\,\mathrm{obs}} - A_G.$$

In [ ]:
# use gspphot

['GSPSPEC-mg_fe', 'GSPSPEC-e_mg_fe']


**Derive absolute magnitude:**

$$M_G = m_G - 5\log_{10}\left(\frac{d}{10\,\mathrm{pc}}\right).$$

**Bolometric correction:**

$$M_{\mathrm{bol}} = M_G + \mathrm{BC}_G.$$

**Compute Luminosity:**

$$L = 10^{0.4\left(M_{\odot,\,\mathrm{bol}} - M_{\mathrm{bol}}\right)}L_\odot.$$